In [ ]:
import os, subprocess, sys, glob, json, shutil, time
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"; os.environ["TQDM_DISABLE"] = "1"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
import torch, hashlib
NGPU = torch.cuda.device_count(); assert NGPU >= 1
print("NGPU", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)], flush=True)


In [ ]:
W = "/kaggle/working"
P = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/recs.jsonl", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "preds"))]
assert len(P) == 1, P; P = P[0]
assert hashlib.md5(open(f"{P}/recs.jsonl", "rb").read()).hexdigest() == "bceef689cda2d6b7b66432970bed1472", "DUNG: recs lech"
WS = f"{W}/pk/thesis"; shutil.copytree(f"{P}/thesis", WS, dirs_exist_ok=True)
VD = f"{W}/data"; os.makedirs(VD, exist_ok=True)
shutil.copy(f"{P}/recs.jsonl", f"{VD}/recs.jsonl")
if os.path.lexists(f"{VD}/images"): os.remove(f"{VD}/images")
os.symlink(f"{P}/images", f"{VD}/images")
recs = [json.loads(l) for l in open(f"{VD}/recs.jsonl")]
assert len(recs) == 622 and all(os.path.exists(f"{VD}/{r['image']}") for r in recs)
PD, OUT = f"{P}/preds", f"{W}/score"; os.makedirs(OUT, exist_ok=True)
from huggingface_hub import snapshot_download
print("UGround:", snapshot_download("osunlp/UGround-V1-2B"), flush=True)
sys.path.insert(0, f"{WS}/harness")
import a11y_inventory as A11Y, score_run as SR
names = set(A11Y._zip().namelist())
co = sum(A11Y.key_for(f"episode_{d['episode_id']}_screenshot_{d['step_id']}.png") in names for d in recs)
rong = sum(len(SR.buttons_of(d)) == 0 for d in recs)
print(f"cay tro nang {co}/622 · rong nut {rong}", flush=True)
assert co == 622 and rong <= 3, "DUNG: thieu cay tro nang"


In [ ]:
env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
lenh = lambda k, out: f"python harness/score_run.py --mode score --grounder uground --preds {PD}/k{k}.jsonl --data-root {VD} --recs-file recs.jsonl --out {out}"
os.makedirs(f"{W}/thu", exist_ok=True)
with open(f"{W}/thu.log", "w") as lg:
    rc = subprocess.run(lenh(0, f"{W}/thu/score_thu.json") + " --n 3", shell=True, cwd=WS, env={**env, "CUDA_VISIBLE_DEVICES": "0"}, stdout=lg, stderr=subprocess.STDOUT).returncode
tl = open(f"{W}/thu.log").read(); print(tl[-1200:], flush=True)
thu = [json.loads(l) for l in open(f"{W}/thu/score_thu_raw.jsonl")]
assert rc == 0 and len(thu) == 3 and f"[dữ liệu] {VD}/recs.jsonl" in tl and "[uground] nạp osunlp/UGround-V1-2B" in tl
assert all(o.get("pred_xy") and o.get("n_buttons", 0) > 0 for o in thu)
print("THU 3 BUOC: DAT", flush=True)
KS = [4, 5, 6, 7, 8]
chuoi = {g: [k for i, k in enumerate(KS) if i % NGPU == g] for g in range(NGPU)}
PS = [subprocess.Popen(["bash", "-c", " && ".join(lenh(k, f"{OUT}/score_k{k}.json") for k in ks)], cwd=WS, env={**env, "CUDA_VISIBLE_DEVICES": str(g)},
                       start_new_session=True, stdout=open(f"{W}/cham_gpu{g}.log", "a"), stderr=subprocess.STDOUT) for g, ks in chuoi.items()]
dem = lambda: [sum(1 for _ in open(f"{OUT}/score_k{k}_raw.jsonl")) if os.path.exists(f"{OUT}/score_k{k}_raw.jsonl") else 0 for k in KS]
t0 = time.time()
while any(p.poll() is None for p in PS):
    time.sleep(120)
    print(f"{(time.time()-t0)/3600:4.2f} h · " + " ".join(f"k{k}:{n}/622" for k, n in zip(KS, dem())), flush=True)
    if time.time() - t0 > 3.8 * 3600:
        [p.terminate() for p in PS]; print("⛔ qua tran gio — dung, giu phan da ghi", flush=True)
print("ma thoat", [p.returncode for p in PS], "dem", dem(), flush=True)


In [ ]:
import zipfile
with zipfile.ZipFile(f"{W}/sa3_cham_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{OUT}/*") + glob.glob(f"{W}/*.log"):
        z.write(p, os.path.basename(p))
print("zip", os.path.getsize(f"{W}/sa3_cham_out.zip") // 1024, "KB", flush=True)
